# 🎬 AIML-03: Large-Scale Two-Tower Recommendation Engine
### Production Candidate Retrieval using TensorFlow Recommenders (TFRS)
**Specialization Track:** AI / Machine Learning  
**Architecture:** Two-Stage Recommender (Stage 1: Two-Tower Retrieval, Stage 2: Deep Ranking)  
**Dataset:** MovieLens 25M with Temporal Split
---
This notebook demonstrates **Step 1 through Step 3** of the Capstone PRD:
1. Temporal Train/Val/Test data cleaning and splitting.
2. Feature Engineering (User interaction history, item popularity, categories).
3. Building, training, and evaluating a Two-Tower Retrieval Model with TFRS.
4. Exporting 32-dimensional User & Item Embeddings for FAISS indexing and Triton serving.

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs

print(f"TensorFlow Version: {tf.__version__}")
print(f"TFRS Version: {tfrs.__version__}")
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Data Ingestion & Temporal Splitting (Step 1)
We enforce a strict **temporal train-validation-test split** (80% / 10% / 10%) based on interaction timestamps.  
This ensures future interactions are not leaked into the training set, mimicking real-world production environments.

In [2]:
from src.data.dataset import prepare_data_pipeline
import os

data_dir = os.path.join(".", "data")
prepare_data_pipeline(data_dir=data_dir)

train_df = pd.read_parquet(os.path.join(data_dir, "train_ratings.parquet"))
val_df = pd.read_parquet(os.path.join(data_dir, "val_ratings.parquet"))
test_df = pd.read_parquet(os.path.join(data_dir, "test_ratings.parquet"))
movies_meta = pd.read_parquet(os.path.join(data_dir, "movies_metadata.parquet"))

print(f"Train set: {len(train_df)} rows")
print(f"Val set:   {len(val_df)} rows")
print(f"Test set:  {len(test_df)} rows")
print(f"Movies in catalog: {len(movies_meta)}")
train_df.head()

## 2. Feature Engineering (Step 2)
We compute key recommendation features across users and movies:
- **User Features:** Total interactions, average rating, rating variance, chronological watch history.
- **Item Features:** Interaction popularity (log-scaled count), average rating, genre multi-hot representations, release year.

In [3]:
from src.data.spark_features import run_feature_pipeline

user_feats, item_feats = run_feature_pipeline(data_dir=data_dir)
print("\n--- User Features Sample ---")
print(user_feats.head(3))
print("\n--- Item Features Sample ---")
print(item_feats.head(3))

## 3. Two-Tower Model Architecture (Step 3)
The Two-Tower model projects user IDs and candidate movie IDs into a shared 32-dimensional embedding space:  
- **User Query Tower:** `StringLookup` -> `Embedding(dim=32)` -> `Dense(64, relu)` -> `Dense(32)` -> `L2_Normalize`  
- **Candidate Item Tower:** `StringLookup` -> `Embedding(dim=32)` -> `Dense(64, relu)` -> `Dense(32)` -> `L2_Normalize`  
- **Loss:** Sampled softmax / In-batch negative cross-entropy via `tfrs.tasks.Retrieval`.

In [4]:
from src.models.two_tower import UserModel, MovieModel, TwoTowerRecommender

unique_movies = movies_meta["movieId"].unique().astype(str)
unique_users = train_df["userId"].unique().astype(str)

# Create candidate dataset
movies_ds = tf.data.Dataset.from_tensor_slices(unique_movies)

# Instantiate Query and Candidate Towers
user_model = UserModel(unique_users, embedding_dimension=32)
movie_model = MovieModel(unique_movies, embedding_dimension=32)

model = TwoTowerRecommender(user_model, movie_model, candidate_dataset=movies_ds)
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.05))
print("Two-Tower Model compiled successfully!")

## 4. Model Training & Evaluation
We train the Two-Tower model on the temporal training dataset and evaluate Top-K retrieval accuracy.

In [5]:
interactions_dict = {
    "userId": train_df["userId"].values.astype(str),
    "movieId": train_df["movieId"].values.astype(str)
}
train_ds = tf.data.Dataset.from_tensor_slices(interactions_dict)
train_ds = train_ds.shuffle(100_000).batch(1024).cache()

print("Starting training...")
history = model.fit(train_ds, epochs=1, verbose=1)
print("Training complete!")

## 5. Artifact Export: Embeddings for FAISS & Triton
We extract the learned 32-dimensional user and movie embeddings and save them as compressed `.npy` files for:
1. High-speed ANN candidate retrieval in **FAISS** (Stage 1).
2. Production serving in **NVIDIA Triton Inference Server**.

In [6]:
models_dir = os.path.join(".", "models")
os.makedirs(models_dir, exist_ok=True)

# Compute Movie Embeddings
movie_batches = movies_ds.batch(512)
movie_embs = []
for b in movie_batches:
    movie_embs.append(movie_model(b).numpy())
movie_embs = np.vstack(movie_embs).astype(np.float32)

# Compute User Embeddings
users_ds = tf.data.Dataset.from_tensor_slices(unique_users).batch(512)
user_embs = []
for b in users_ds:
    user_embs.append(user_model(b).numpy())
user_embs = np.vstack(user_embs).astype(np.float32)

print(f"User Embeddings shape: {user_embs.shape}")
print(f"Movie Embeddings shape: {movie_embs.shape}")